# Wang2015 — Vapor de agua

Estima vapor d’água integrado na coluna atmosférica (PWV), com saída em **g/cm²**. Wang utiliza grupos globais de NDVI e regressão local.

## Precisa de

**Temperaturas de brilho — obrigatórias:**

- **BT10:** imagem de temperatura de brilho TOA da banda 10, em **K (kelvin)**.
- **BT11:** imagem de temperatura de brilho TOA da banda 11, em **K (kelvin)**.

Não fornecer DN, radiância, LST ou temperaturas em Celsius nesses dois parâmetros.

**Bandas para o NDVI — obrigatórias nesta rota:**

- **Vermelho:** imagem calibrada em radiância TOA, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Infravermelho próximo (NIR):** imagem calibrada na **mesma grandeza e unidade do vermelho**.

O NDVI é calculado internamente. É possível substituir as duas bandas por `ndvi_image` preparado, **adimensional, em [-1,1]**; não combinar as entradas.

**Qualidade e suporte espacial:**

- **Máscara de exclusão:** imagem **booleana**, sem unidade; `True` rejeita o pixel. Este notebook usa `exclude_water=True` para excluir também água, além das rejeições de qualidade do leitor.
- **`window_size`:** lado da janela local, em **pixels**, inteiro ímpar ≥3. O tamanho em metros depende da resolução da grade; o tamanho demonstrativo e as demais opções estão na tabela.

- **`ndvi_range`:** extremos mínimo/máximo do NDVI, **adimensionais**, da cena inteira. O exemplo os calcula com `scene_ndvi_range`; ao processar blocos, reutilizar os mesmos extremos globais e prever halo.

PWV é a **saída** do método, não uma imagem que você precisa fornecer como entrada.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo usa o leitor de cena **Landsat 8/9 Collection 2 Level 1**. Coloque na pasta da cena:

- **B4:** arquivo original do vermelho, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B5:** arquivo original do NIR, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos inteiros de qualidade, sem unidade física; o leitor interpreta os bits para montar a máscara.
- **QA_RADSAT:** códigos inteiros de saturação e oclusão de terreno, sem unidade física; complementam a máscara.

Essa lista corresponde ao leitor usado no notebook; uma chamada direta à função exige somente as entradas indicadas em “Precisa de”. B2 e SZA são opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos. Não passar os arquivos QA brutos como máscara booleana.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="radiance"` devolve as bandas ópticas em **radiância TOA, em W m⁻² sr⁻¹ µm⁻¹**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="reflectance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = water_vapor(
    # Método
    method='wang-2015',

    # Entradas para o NDVI
    red=opticas[4],
    nir=opticas[5],

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Configuração do estimador
    window_size=21,
    group_count=3,
    ndvi_range=limites_ndvi,

    # Processamento em lotes
    chunk_rows=8,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"wang-2015"` | Seletor | Usar "wang-2015" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `red` | Obrigatório nesta rota; dispensável com NDVI/emissividades prontos | `None` | W m⁻² sr⁻¹ µm⁻¹ ou adimensional | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `nir` | Obrigatório nesta rota; dispensável com NDVI/emissividades prontos | `None` | W m⁻² sr⁻¹ µm⁻¹ ou adimensional | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `brightness_band_10` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B10, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `brightness_band_11` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B11, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `window_size` | Opcional | `None → 21` | Pixels | Lado da janela quadrada, inteiro ímpar ≥3. O suporte em metros depende da resolução da grade; usar halo ao processar blocos. A janela 21 é uma adaptação operacional do suporte 20 do artigo. |
| `group_count` | Opcional somente em Wang | `3` | Contagem | Número inteiro de grupos NDVI ≥1. Não se aplica a Ren. |
| `ndvi_range` | Opcional; global ao processar blocos | `None` | (NDVI mínimo, máximo) | Extremos de toda a cena, obtidos com scene_ndvi_range. Reutilizar em todos os blocos; None determina os extremos da entrada atual. |
| `chunk_rows` | Opcional | `8` | Linhas de pixels | Inteiro ≥1; controla memória/lotes sem mudar as equações. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — NDVI preparado

Substitui somente vermelho/NIR; manter as outras configurações da chamada principal.

```python
resultado = water_vapor(
    # Método
    method='wang-2015',

    # Entradas para o NDVI
    ndvi_image=ndvi_preparado,

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Configuração do estimador
    window_size=21,
    group_count=3,
    ndvi_range=limites_ndvi,

    # Processamento em lotes
    chunk_rows=8,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"wang-2015"` | Seletor | Usar "wang-2015" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `ndvi_image` | Condicional: alternativa ao par vermelho/NIR | `None` | Adimensional, [-1,1] | NDVI já preparado e alinhado; omitir as duas bandas ópticas. A convenção radiométrica deve respeitar a fonte do método. Não combinar NDVI preparado e vermelho/NIR. |
| `brightness_band_10` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B10, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `brightness_band_11` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B11, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `window_size` | Opcional | `None → 21` | Pixels | Lado da janela quadrada, inteiro ímpar ≥3. O suporte em metros depende da resolução da grade; usar halo ao processar blocos. A janela 21 é uma adaptação operacional do suporte 20 do artigo. |
| `group_count` | Opcional somente em Wang | `3` | Contagem | Número inteiro de grupos NDVI ≥1. Não se aplica a Ren. |
| `ndvi_range` | Opcional; global ao processar blocos | `None` | (NDVI mínimo, máximo) | Extremos de toda a cena, obtidos com scene_ndvi_range. Reutilizar em todos os blocos; None determina os extremos da entrada atual. |
| `chunk_rows` | Opcional | `8` | Linhas de pixels | Inteiro ≥1; controla memória/lotes sem mudar as equações. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import water_vapor
from pylstemp.workflows import scene_ndvi_range
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/wang_2015")
saida.mkdir(parents=True, exist_ok=True)
janela_estimador = 21
linhas_por_lote = 8
grupos_ndvi = 3

## Ler e calibrar a imagem

`LandsatScene` confere os arquivos/grade da cena e le o MTL. `calibrated()` retorna bandas opticas, BT10/BT11 em K e mascara QA booleana. As entradas deste exemplo sao radiancias opticas TOA.

### Máscara usada neste exemplo

`mask` é uma máscara **de exclusão**, não uma máscara de seleção: `True` rejeita o pixel e `False` permite o cálculo. O leitor usa QA_PIXEL para fill, nuvem dilatada, cirrus de alta confiança, nuvem, sombra e neve; QA_RADSAT para saturação óptica B2/B4/B5 e oclusão de terreno; e verifica SZA quando disponível. A opção `exclude_water=True` também exclui água (bit7 de QA_PIXEL). Valores não finitos ou fora do domínio podem ser rejeitados adicionalmente pelo cálculo. `mask=None` em uma chamada direta não cria automaticamente uma máscara de nuvens.


In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(optical_quantity="radiance", exclude_water=True)

## Aplicar o metodo

r2>=.97; referencias TIGR e extrapolacoes descritas na ficha. Ao processar blocos, usar limites globais e halo; o workflow Du faz essa gestao. Ren possui outro notebook.

In [ ]:
limites_ndvi = scene_ndvi_range(cena, optical_quantity='radiance')
imagem = water_vapor(
    # Método
    method='wang-2015',

    # Entradas para o NDVI
    red=opticas[4],
    nir=opticas[5],

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Configuração do estimador
    window_size=janela_estimador,
    group_count=grupos_ndvi,
    ndvi_range=limites_ndvi,

    # Processamento em lotes
    chunk_rows=linhas_por_lote,

    # Exclusão de pixels
    mask=mascara,
)


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
cena.write_geotiff(saida / "wang_2015.tif", imagem, units='g/cm2')
cena.close()
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).